In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Camada Bronze — Ingestão dos dados brutos do Airbnb Rio de Janeiro
# MAGIC 
# MAGIC **Objetivo desta etapa:** carregar os arquivos CSV mensais exatamente como vieram da fonte,
# MAGIC preservando rastreabilidade (arquivo de origem, mês de referência, data de ingestão).
# MAGIC Nenhuma limpeza ou transformação de conteúdo acontece aqui — isso é responsabilidade da camada Silver.
# MAGIC
# MAGIC **Fonte dos dados:** Kaggle — Rio de Janeiro Airbnb Open Data
# MAGIC https://www.kaggle.com/datasets/allanbruno/airbnb-rio-de-janeiro
# MAGIC
# MAGIC **Licença:** CC0: Public Domain
# MAGIC Datasets baseados em Inside Airbnb costumam exigir atribuição à fonte original.]

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Configuração inicial

# COMMAND ----------

from pyspark.sql.functions import regexp_extract, current_timestamp, to_date, col, lower, create_map, lit, concat
from itertools import chain

CAMINHO_ARQUIVOS_BRUTOS = "/Volumes/mvp_airbnb_rj/bronze/raw_files/*.csv"

CATALOGO_DESTINO = "mvp_airbnb_rj"
SCHEMA_DESTINO = "bronze"
TABELA_DESTINO = "listings_bronze"

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Leitura de todos os arquivos mensais de uma vez
# MAGIC
# MAGIC Foi utilizado wildcard (`*.csv`) para ler todos os arquivos da pasta numa única operação,
# MAGIC em vez de fazer um load separado por mês. O Spark distribui essa leitura automaticamente.
# MAGIC
# MAGIC Cada CSV do Inside Airbnb costuma ter ~108 colunas. Nesta camada, mantemos TODAS as colunas
# MAGIC como vieram (princípio da Bronze: preservar o dado bruto). A seleção de colunas relevantes
# MAGIC acontece só na Silver.

# COMMAND ----------

df_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")  
    .option("multiLine", "true")    
    .option("escape", "\"")
    .csv(CAMINHO_ARQUIVOS_BRUTOS)
)

print(f"Total de linhas carregadas (todos os meses): {df_bronze.count()}")
print(f"Total de colunas: {len(df_bronze.columns)}")


# COMMAND ----------

# Extrai o nome do mês (letras) e o ano (4 dígitos) do nome do arquivo.
# Exemplo: ".../janeiro2019.csv" -> mes_nome = "janeiro", ano = "2019"
#
# NOTA: usamos col("_metadata.file_path") em vez de input_file_name() porque, em ambientes
# com Unity Catalog (como o Databricks Free Edition, que roda em compute serverless),
# a função input_file_name() foi descontinuada. A coluna oculta "_metadata" é exposta
# automaticamente pelo Spark em toda leitura de arquivo e contém informações como
# file_path, file_name, file_size e file_modification_time.
df_bronze_com_metadados = (
    df_bronze
    .withColumn("arquivo_origem", col("_metadata.file_path"))
    .withColumn("mes_nome", lower(regexp_extract(col("arquivo_origem"), r"([a-zA-Zç]+)(\d{4})\.csv", 1)))
    .withColumn("ano", regexp_extract(col("arquivo_origem"), r"([a-zA-Zç]+)(\d{4})\.csv", 2))
)


mapa_meses = {
    "janeiro": "01", "fevereiro": "02", "marco": "03", "março": "03",
    "abril": "04", "maio": "05", "junho": "06", "julho": "07",
    "agosto": "08", "setembro": "09", "outubro": "10",
    "novembro": "11", "dezembro": "12",
    # Aliases para nomes de arquivo com caracteres faltando (achado real na fonte de dados):
    "maro": "03",       # variante corrompida de "março" (sem o "ç")
    "novrmbro": "11",   # variante corrompida de "novembro" (sem o "e")
}

# Cria uma expressão de mapeamento utilizável em coluna do Spark (create_map espera pares chave-valor "achatados")
mapa_spark = create_map([lit(x) for pair in mapa_meses.items() for x in pair])

df_bronze_com_metadados = (
    df_bronze_com_metadados
    .withColumn("mes_numero", mapa_spark[col("mes_nome")])
    .withColumn(
        "mes_referencia",
        # concatena ano + mês numérico no padrão "YYYY-MM", ex: "2019-01"
        # IMPORTANTE: no PySpark, o operador "+" tenta fazer soma numérica, não concatenação de texto.
        # Por isso a função concat(), que é a forma correta de juntar strings em colunas.
        concat(col("ano"), lit("-"), col("mes_numero"))
    )
    .withColumn("data_ingestao", current_timestamp())
)

# Checagem rápida: quantos arquivos/meses distintos foram identificados?
# Se aparecer algum "null" em mes_referencia, é sinal de que algum nome de arquivo não bateu
# com o padrão esperado (ex: acento diferente, mês abreviado, etc.) — inspecione arquivo_origem nesses casos.
df_bronze_com_metadados.select("arquivo_origem", "mes_nome", "ano", "mes_referencia").distinct().orderBy("mes_referencia").show(50, truncate=False)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Persistir como tabela Delta na camada Bronze
# MAGIC
# MAGIC Usamos Delta Lake (formato nativo do Databricks) para já ter transações ACID
# MAGIC e possibilidade de time travel, caso precise auditar cargas anteriores.

# COMMAND ----------

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO_DESTINO}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO_DESTINO}.{SCHEMA_DESTINO}")

(
    df_bronze_com_metadados
    .write
    .format("delta")
    .mode("overwrite")  
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOGO_DESTINO}.{SCHEMA_DESTINO}.{TABELA_DESTINO}")
)

print(f"Tabela Bronze criada: {CATALOGO_DESTINO}.{SCHEMA_DESTINO}.{TABELA_DESTINO}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Validação final da carga

# COMMAND ----------

df_validacao = spark.table(f"{CATALOGO_DESTINO}.{SCHEMA_DESTINO}.{TABELA_DESTINO}")

print("Amostra dos dados carregados:")
display(df_validacao.select("id", "price", "neighbourhood", "mes_referencia", "arquivo_origem").limit(10))

print(f"\nTotal de registros na Bronze: {df_validacao.count()}")
print(f"Total de meses distintos: {df_validacao.select('mes_referencia').distinct().count()}")
